# NegMerge Reproduction — Scenario B (CIFAR-10 / CUB-200 / Tiny ImageNet)

Reproduces the 10% random forgetting scenario for ResNet-18 / VGG-16.

**Supported (BACKBONE, DATASET) combos:**
- (resnet18, cifar10)      — Table 2,  NegMerge Avg. Gap ≈ 1.07
- (vgg16,    cifar10)      — Table B10, NegMerge Avg. Gap ≈ 1.50

## Cell 0 — Configuration

In [1]:
# Cell 0: Global config — the ONLY place you set backbone / dataset.
# ─────────────────────────────────────────────────────────────────
BACKBONE = "resnet18"       # "resnet18"  |  "vgg16"
DATASET  = "cifar10"
# ─────────────────────────────────────────────────────────────────

VALID_COMBOS = {
    ("resnet18", "cifar10"),
    ("vgg16",    "cifar10"),
}
assert (BACKBONE, DATASET) in VALID_COMBOS, \
    f"Invalid combo: ({BACKBONE}, {DATASET}). Supported: {sorted(VALID_COMBOS)}"

# ── Paper targets, keyed by (backbone, dataset) ────────────────────────────
PAPER_TARGETS = {
    ("resnet18", "cifar10"): {
        "retrain_acc_dtest": 94.26, "retrain_acc_df": 94.76,
        "retrain_acc_dr":   100.00, "retrain_mia":    12.88,
        "negmerge_avg_gap":   1.07,
        "ta_single_best_avg_gap": 1.62,
        "uniform_merge_avg_gap":  1.75,
    },
    ("vgg16", "cifar10"): {
        "retrain_acc_dtest": 93.06, "retrain_acc_df": 94.02,
        "retrain_acc_dr":    99.99, "retrain_mia":    10.36,
        "negmerge_avg_gap":   1.50,
        "ta_single_best_avg_gap": 1.64,
        "uniform_merge_avg_gap":  None,
    },
}
targets = PAPER_TARGETS[(BACKBONE, DATASET)]

NUM_CLASSES = {"cifar10": 10}[DATASET]

print(f"BACKBONE    = {BACKBONE}")
print(f"DATASET     = {DATASET}")
print(f"NUM_CLASSES = {NUM_CLASSES}")
print(f"Paper targets: {targets}")


BACKBONE    = resnet18
DATASET     = cifar10
NUM_CLASSES = 10
Paper targets: {'retrain_acc_dtest': 94.26, 'retrain_acc_df': 94.76, 'retrain_acc_dr': 100.0, 'retrain_mia': 12.88, 'negmerge_avg_gap': 1.07, 'ta_single_best_avg_gap': 1.62, 'uniform_merge_avg_gap': 1.75}


## Cell 1 — Mount Drive, set paths, install deps

In [2]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    _drive_root = '/content/drive/MyDrive/negmerge_reproduction'
else:
    _drive_root = os.path.abspath('./negmerge_reproduction')

# Per-combo directory, e.g. resnet18_cifar10/
RUN_TAG   = f"{BACKBONE}_{DATASET}"
DRIVE_DIR = os.path.join(_drive_root, RUN_TAG)
POOL_DIR  = os.path.join(DRIVE_DIR, 'pool')
os.makedirs(POOL_DIR, exist_ok=True)

THETA_PRE_PATH     = os.path.join(DRIVE_DIR, 'theta_pre.pt')
THETA_RETRAIN_PATH = os.path.join(DRIVE_DIR, 'theta_retrain.pt')
FORGET_IDX_PATH    = os.path.join(DRIVE_DIR, 'forget_indices_seed1.npy')
RETAIN_IDX_PATH    = os.path.join(DRIVE_DIR, 'retain_indices_seed1.npy')
TAU_MERGED_PATH    = os.path.join(DRIVE_DIR, 'tau_merged.pt')
TAU_UNIFORM_PATH   = os.path.join(DRIVE_DIR, 'tau_uniform.pt')
RESULTS_CSV        = os.path.join(DRIVE_DIR, 'results.csv')

# Shared dataset caches.
DATA_ROOT = os.path.join(_drive_root, 'data')
os.makedirs(DATA_ROOT, exist_ok=True)
CIFAR_DATA_DIR    = os.path.join(DATA_ROOT, 'cifar10')
os.makedirs(CIFAR_DATA_DIR, exist_ok=True)

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'scikit-learn>=1.3', 'numpy>=1.24'], check=True)

print(f'RUN_TAG   = {RUN_TAG}')
print(f'DRIVE_DIR = {DRIVE_DIR}')
print(f'POOL_DIR  = {POOL_DIR}')
print(f'DATA_ROOT = {DATA_ROOT}')


Mounted at /content/drive
RUN_TAG   = resnet18_cifar10
DRIVE_DIR = /content/drive/MyDrive/negmerge_reproduction/resnet18_cifar10
POOL_DIR  = /content/drive/MyDrive/negmerge_reproduction/resnet18_cifar10/pool
DATA_ROOT = /content/drive/MyDrive/negmerge_reproduction/data


## Cell 2 — Imports, seeding, dataset-specific loading

Branches on `DATASET` to load CIFAR-10.

In [3]:
# Cell 2: imports + seed helpers + dataset-specific loading

import gc, time, json, math, random, tarfile, zipfile, urllib.request
from pathlib import Path
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, Dataset
import torchvision
import torchvision.transforms as transforms

DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      ('mps' if torch.backends.mps.is_available() else 'cpu'))
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


# ── Normalisation stats ────────────────────────────────────────────────────
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)


# ── Dataset loading, branched on DATASET ───────────────────────────────────
if DATASET == "cifar10":
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    trainset_aug   = torchvision.datasets.CIFAR10(CIFAR_DATA_DIR, train=True,  download=True,  transform=train_transform)
    trainset_plain = torchvision.datasets.CIFAR10(CIFAR_DATA_DIR, train=True,  download=False, transform=eval_transform)
    testset        = torchvision.datasets.CIFAR10(CIFAR_DATA_DIR, train=False, download=False, transform=eval_transform)
else:
    raise ValueError(f'Unknown DATASET: {DATASET}')

print(f'Train: {len(trainset_aug)}  Test: {len(testset)}  (DATASET={DATASET})')


Device: cuda
GPU: Tesla T4
Train: 50000  Test: 10000  (DATASET=cifar10)


## Cell 3 — Model definitions (backbone × dataset)

`make_model()` is the single factory used by every downstream cell. It branches on `(BACKBONE, DATASET)` to pick the right stem / classifier:

- **resnet18 / cifar10** — CIFAR variant: 3×3 stem, no maxpool (kuangliu).
- **vgg16 / cifar10** — torchvision VGG-16, classifier head replaced.


In [4]:
# ── ResNet-18 (CIFAR variant; 3×3 stem, no maxpool) ───────────────────────
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes),
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

class ResNet18CIFAR(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.in_planes = 64
        self.conv1  = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64,  2, 1)
        self.layer2 = self._make_layer(128, 2, 2)
        self.layer3 = self._make_layer(256, 2, 2)
        self.layer4 = self._make_layer(512, 2, 2)
        self.linear = nn.Linear(512, num_classes)
    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes * BasicBlock.expansion
        return nn.Sequential(*layers)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out); out = self.layer2(out)
        out = self.layer3(out); out = self.layer4(out)
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.linear(out)

# ── VGG-16-BN (CIFAR variant) ─────────────────────────────────────────────
def VGG16CIFAR(num_classes=10):
    """VGG-16 with BatchNorm, 1x1 adaptive pool, and a CIFAR-sized classifier.

    Plain torchvision VGG-16 (no BN) diverges during warmup on CIFAR-10 because
    there's no normalisation to keep activations bounded. vgg16_bn is stable.

    The default torchvision head expects a 7x7 feature map from 224x224 input.
    CIFAR-10 is 32x32, so features reduce to 1x1 after the stride-32 conv
    stack. Upsampling 1x1 -> 7x7 and running it through a 25088-dim FC is
    wasteful and ill-conditioned; collapse the head to a 512-dim path instead.
    """
    model = torchvision.models.vgg16_bn(weights=None)
    model.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    model.classifier = nn.Sequential(
        nn.Linear(512, 512), nn.ReLU(True), nn.Dropout(0.5),
        nn.Linear(512, 512), nn.ReLU(True), nn.Dropout(0.5),
        nn.Linear(512, num_classes),
    )
    return model


# ── Factory ────────────────────────────────────────────────────────────────
def make_model(backbone=None, dataset=None, num_classes=None):
    b = backbone if backbone is not None else BACKBONE
    d = dataset  if dataset  is not None else DATASET
    nc = num_classes if num_classes is not None else NUM_CLASSES
    if   (b, d) == ("resnet18", "cifar10"):       return ResNet18CIFAR(nc)
    elif (b, d) == ("vgg16",    "cifar10"):       return VGG16CIFAR(nc)
    else: raise ValueError(f'Unsupported combo: ({b}, {d})')

# Sanity: param count
_m = make_model(); _n = sum(p.numel() for p in _m.parameters())
print(f'{BACKBONE}/{DATASET} params: {_n/1e6:.2f}M')
del _m

resnet18/cifar10 params: 11.17M


## Cell 4 — Hyperparameter configs + training / eval helpers

Per-(backbone, dataset) hyperparameters for Stage 1, Retrain, and the Pool live in `STAGE1_CONFIGS`, `RETRAIN_CONFIGS`, and `POOL_CONFIGS`. See guide §3, §5, §8.6.

`train_from_scratch` — Stage 1 + Retrain: SGD + MultiStepLR + linear warmup.
`finetune_pool_member` — 27-model pool: SGD + CosineAnnealing, no augmentation.

In [5]:
# Per-(backbone, dataset) config dicts.
# Confidence annotations:
#  specified in NegMerge / SalUn / Jia et al. papers

STAGE1_CONFIGS = {
    ("resnet18", "cifar10"): dict(
        lr=0.1, epochs=182, batch_size=256, weight_decay=5e-4,
        milestones=(91, 136), warmup=1, pretrained=False,
    ),
    ("vgg16", "cifar10"): dict(                   # vgg16_bn needs a gentler LR
        lr=0.05, epochs=182, batch_size=64, weight_decay=5e-4,
        milestones=(91, 136), warmup=5, pretrained=False,
    ),

}

RETRAIN_CONFIGS = {
    ("resnet18", "cifar10"):       dict(lr=0.1,  epochs=160, batch_size=256, weight_decay=5e-4, milestones=(80, 120),  warmup=1),
    ("vgg16",    "cifar10"):       dict(lr=0.05, epochs=160, batch_size=64,  weight_decay=5e-4, milestones=(80, 120),  warmup=5),
}

POOL_CONFIGS = {
    ("resnet18", "cifar10"): dict(                # NegMerge Appendix A
        lr=0.1, batch_size=256,
        epochs_grid=(40, 50, 60),
        wd_grid=(1e-4, 5e-5, 1e-5),
        ls_grid=(0.0, 0.05, 0.1),
    ),
    ("vgg16", "cifar10"): dict(                   # NegMerge Appendix A
        lr=0.01, batch_size=64,
        epochs_grid=(40, 50, 60),
        wd_grid=(1e-4, 5e-5, 1e-5),
        ls_grid=(0.0, 0.05, 0.1),
    ),
}

cfg_stage1  = STAGE1_CONFIGS[(BACKBONE, DATASET)]
cfg_retrain = RETRAIN_CONFIGS[(BACKBONE, DATASET)]
cfg_pool    = POOL_CONFIGS[(BACKBONE, DATASET)]

STAGE1_BS     = cfg_stage1['batch_size']
STAGE1_WARMUP = cfg_stage1['warmup']
POOL_LR       = cfg_pool['lr']
POOL_BS       = cfg_pool['batch_size']

print(f'cfg_stage1  = {cfg_stage1}')
print(f'cfg_retrain = {cfg_retrain}')
print(f'cfg_pool    = {cfg_pool}')


def make_loader(dataset, indices=None, batch_size=256, shuffle=True, num_workers=2):
    ds = dataset if indices is None else Subset(dataset, indices)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=num_workers, pin_memory=True, drop_last=False)

@torch.no_grad()
def evaluate_accuracy(model, loader, device=DEVICE):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / total

def train_from_scratch(
    train_loader, test_loader,
    epochs=182, lr=0.1, momentum=0.9, weight_decay=5e-4,
    milestones=(91, 136), gamma=0.1, warmup_epochs=1,
    label_smoothing=0.0, log_every=10, device=DEVICE,
):
    """Stage-1 / Retrain recipe: SGD + MultiStepLR + linear warmup.
    Uses the current (BACKBONE, DATASET) via make_model()."""
    model   = make_model().to(device)
    opt     = torch.optim.SGD(model.parameters(), lr=lr,
                               momentum=momentum, weight_decay=weight_decay)
    sch     = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=list(milestones), gamma=gamma)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    n_steps = len(train_loader)

    for epoch in range(epochs):
        model.train()
        running = 0.0
        for i, (x, y) in enumerate(train_loader):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            if epoch < warmup_epochs:
                warm_lr = lr * (i + 1 + epoch * n_steps) / max(1, warmup_epochs * n_steps)
                for g in opt.param_groups: g['lr'] = warm_lr
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward(); opt.step()
            running += loss.item() * y.size(0)
        if epoch >= warmup_epochs:
            sch.step()
        if (epoch + 1) % log_every == 0 or epoch == epochs - 1:
            acc = evaluate_accuracy(model, test_loader, device)
            print(f'  epoch {epoch+1:3d}/{epochs}  '
                  f'loss {running/len(train_loader.dataset):.4f}  '
                  f'test {acc:.2f}%  '
                  f'lr {opt.param_groups[0]["lr"]:.5f}')
    return model

def finetune_pool_member(
    pretrained_state, train_loader,
    epochs=40, lr=None, momentum=0.9,
    weight_decay=1e-4, label_smoothing=0.0, device=DEVICE,
):
    """Pool fine-tuning: SGD + CosineAnnealing.
    LR defaults to the (backbone, dataset)-specific POOL_LR from Cell 4."""
    if lr is None:
        lr = POOL_LR
    model = make_model().to(device)
    model.load_state_dict(pretrained_state)
    opt     = torch.optim.SGD(model.parameters(), lr=lr,
                               momentum=momentum, weight_decay=weight_decay)
    sch     = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    for _ in range(epochs):
        model.train()
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            opt.step()
        sch.step()
    return model


cfg_stage1  = {'lr': 0.1, 'epochs': 182, 'batch_size': 256, 'weight_decay': 0.0005, 'milestones': (91, 136), 'warmup': 1, 'pretrained': False}
cfg_retrain = {'lr': 0.1, 'epochs': 160, 'batch_size': 256, 'weight_decay': 0.0005, 'milestones': (80, 120), 'warmup': 1}
cfg_pool    = {'lr': 0.1, 'batch_size': 256, 'epochs_grid': (40, 50, 60), 'wd_grid': (0.0001, 5e-05, 1e-05), 'ls_grid': (0.0, 0.05, 0.1)}


## Cell 5 — [Stage 1] Train θ_pre on full training set

Uses `cfg_stage1` for lr / epochs / milestones / warmup. Skips if `theta_pre.pt` already exists in `DRIVE_DIR`.

In [6]:
STAGE1_EPOCHS = cfg_stage1['epochs']

# Stale-checkpoint guard: a prior VGG-16 (no-BN) run could have saved a
# collapsed theta_pre.pt (~10% test acc). Detect and remove it before skipping.
if BACKBONE == "vgg16" and os.path.exists(THETA_PRE_PATH):
    _stale = make_model().to(DEVICE)
    _stale.load_state_dict(torch.load(THETA_PRE_PATH, map_location=DEVICE))
    _stale_acc = evaluate_accuracy(_stale, make_loader(testset, batch_size=512, shuffle=False))
    print(f'VGG stale-check: existing θ_pre Acc D_test = {_stale_acc:.2f}%')
    if _stale_acc < 50.0:
        print(f'  → below 50% floor; deleting {THETA_PRE_PATH} and retraining.')
        os.remove(THETA_PRE_PATH)
    del _stale
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

if os.path.exists(THETA_PRE_PATH):
    print(f'Found θ_pre at {THETA_PRE_PATH} — skipping training.')
    theta_pre_state = torch.load(THETA_PRE_PATH, map_location='cpu')
else:
    set_seed(1)
    full_train_loader = make_loader(trainset_aug,  batch_size=cfg_stage1['batch_size'], shuffle=True)
    test_loader       = make_loader(testset,       batch_size=512, shuffle=False)
    t0 = time.time()
    model = train_from_scratch(
        full_train_loader, test_loader,
        epochs=cfg_stage1['epochs'], lr=cfg_stage1['lr'], momentum=0.9,
        weight_decay=cfg_stage1['weight_decay'],
        milestones=cfg_stage1['milestones'], gamma=0.1,
        warmup_epochs=cfg_stage1['warmup'],
        label_smoothing=0.0, log_every=10,
    )
    print(f'Stage 1 done in {(time.time()-t0)/60:.1f} min')
    theta_pre_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    torch.save(theta_pre_state, THETA_PRE_PATH)
    print(f'Saved θ_pre → {THETA_PRE_PATH}')
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# ── Sanity check ──────────────────────────────────────────────────────────
_m = make_model().to(DEVICE); _m.load_state_dict(theta_pre_state)
acc_dtest = evaluate_accuracy(_m, make_loader(testset, batch_size=512, shuffle=False))
gap = abs(acc_dtest - targets["retrain_acc_dtest"])
status = "within 0.5% of paper" if gap < 0.5 else "deviates from paper (may still be fine)"
print(f"θ_pre Acc D_test = {acc_dtest:.2f}%  |  Paper target ≈ {targets['retrain_acc_dtest']}%  |  {status}")

# Per-dataset minimum-accuracy floors (guide §3.3). Sanity floor only — below
# this, something is badly wrong (bad pretraining load, wrong stem, etc.).
FLOOR = {
    ("resnet18", "cifar10"):       93.0,
    ("vgg16",    "cifar10"):       92.0,
}[(BACKBONE, DATASET)]
assert acc_dtest >= FLOOR, (
    f'Accuracy {acc_dtest:.2f}% below floor {FLOOR}% — see guide §9 / §C1 / §C2.'
)
del _m; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


Found θ_pre at /content/drive/MyDrive/negmerge_reproduction/resnet18_cifar10/theta_pre.pt — skipping training.
θ_pre Acc D_test = 95.01%  |  Paper target ≈ 94.26%  |  deviates from paper (may still be fine)


## Cell 6 — [Stage 2] Forget / retain splits (10% random)

Sizes scale with the training-set size:
- CIFAR-10: 5,000 / 45,000


In [7]:
SPLIT_SEED = 1
N_TRAIN    = len(trainset_aug)
N_FORGET   = int(0.10 * N_TRAIN)

if os.path.exists(FORGET_IDX_PATH) and os.path.exists(RETAIN_IDX_PATH):
    forget_indices = np.load(FORGET_IDX_PATH)
    retain_indices = np.load(RETAIN_IDX_PATH)
    print(f'Loaded splits from Drive (seed={SPLIT_SEED}).')
else:
    rng = np.random.default_rng(SPLIT_SEED)
    forget_indices = np.sort(rng.choice(N_TRAIN, size=N_FORGET, replace=False))
    retain_indices = np.setdiff1d(np.arange(N_TRAIN), forget_indices)
    np.save(FORGET_IDX_PATH, forget_indices)
    np.save(RETAIN_IDX_PATH, retain_indices)
    print(f'Saved splits (seed={SPLIT_SEED}).')

assert len(forget_indices) == N_FORGET
assert len(retain_indices) == N_TRAIN - N_FORGET
assert len(np.intersect1d(forget_indices, retain_indices)) == 0
print(f'|D_f| = {len(forget_indices)}  |D_r| = {len(retain_indices)}  '
      f'(DATASET={DATASET}, N_TRAIN={N_TRAIN})')


Loaded splits from Drive (seed=1).
|D_f| = 5000  |D_r| = 45000  (DATASET=cifar10, N_TRAIN=50000)


## Cell 7 — [Retrain baseline] Train from scratch on D_r only

Uses `cfg_retrain` for lr / epochs / milestones / warmup. This is the used to calcualte Avg. Gap. Reuses make_model() so it inherits the right architecture for the current (BACKBONE, DATASET) combo.

In [8]:
RETRAIN_EPOCHS = cfg_retrain['epochs']

if os.path.exists(THETA_RETRAIN_PATH):
    print(f'Found θ_retrain at {THETA_RETRAIN_PATH} — skipping training.')
    theta_retrain_state = torch.load(THETA_RETRAIN_PATH, map_location='cpu')
else:
    set_seed(2)
    retain_loader_aug = make_loader(trainset_aug, indices=retain_indices,
                                    batch_size=cfg_retrain['batch_size'], shuffle=True)
    test_loader       = make_loader(testset, batch_size=512, shuffle=False)
    t0 = time.time()
    model = train_from_scratch(
        retain_loader_aug, test_loader,
        epochs=cfg_retrain['epochs'], lr=cfg_retrain['lr'], momentum=0.9,
        weight_decay=cfg_retrain['weight_decay'],
        milestones=cfg_retrain['milestones'], gamma=0.1,
        warmup_epochs=cfg_retrain['warmup'],
        label_smoothing=0.0, log_every=10,
    )
    print(f'Retrain done in {(time.time()-t0)/60:.1f} min')
    theta_retrain_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    torch.save(theta_retrain_state, THETA_RETRAIN_PATH)
    print(f'Saved θ_retrain → {THETA_RETRAIN_PATH}')
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# Sanity check: report all three splits
_m = make_model().to(DEVICE); _m.load_state_dict(theta_retrain_state)
_splits = {
    'D_r':    make_loader(trainset_plain, indices=retain_indices, batch_size=512, shuffle=False),
    'D_f':    make_loader(trainset_plain, indices=forget_indices, batch_size=512, shuffle=False),
    'D_test': make_loader(testset, batch_size=512, shuffle=False),
}
retrain_accs = {}
for name, ldr in _splits.items():
    a = evaluate_accuracy(_m, ldr)
    retrain_accs[name] = a
    t = targets.get(f'retrain_acc_{name.lower().replace("_","")}')
    print(f'θ_retrain Acc {name}: {a:.2f}%' + (f'  (paper ≈ {t}%)' if t else ''))
del _m; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


Found θ_retrain at /content/drive/MyDrive/negmerge_reproduction/resnet18_cifar10/theta_retrain.pt — skipping training.
θ_retrain Acc D_r: 100.00%  (paper ≈ 100.0%)
θ_retrain Acc D_f: 94.40%  (paper ≈ 94.76%)
θ_retrain Acc D_test: 94.43%  (paper ≈ 94.26%)


## Cell 8 — [Stage 3] Fine-tune the 27-model pool (resumable)

Grid from `cfg_pool`: `epochs_grid × wd_grid × ls_grid` = 27 models.

- CIFAR-10 (RN-18 / VGG-16): epochs ∈ {40, 50, 60}

Augmentation ENABLED during pool training to ensure task-vector diversity — without it the pool is homogeneous (all members memorise D_f identically, yielding 99%+ sparsity on CIFAR-10 after sign-consensus). This deviates from NegMerge Appendix A's "no augmentation" note; see reproduction guide.

CosineAnnealing scheduler. Resumable — scans `pool/` on entry.

In [9]:
REDUCED_POOL = False   # True = 8-model dry run

full_grid = [
    (epochs, wd, ls)
    for epochs in cfg_pool['epochs_grid']
    for wd     in cfg_pool['wd_grid']
    for ls     in cfg_pool['ls_grid']
]
# Dry-run grid: corners of the cube (2×2×2 = 8 models).
_eg, _wg, _lg = cfg_pool['epochs_grid'], cfg_pool['wd_grid'], cfg_pool['ls_grid']
reduced_grid = [
    (epochs, wd, ls)
    for epochs in (_eg[0], _eg[-1])
    for wd     in (_wg[0], _wg[-1])
    for ls     in (_lg[0], _lg[-1])
]
grid   = reduced_grid if REDUCED_POOL else full_grid
N_POOL = len(grid)
print(f'Pool grid size: {N_POOL}  (REDUCED_POOL={REDUCED_POOL})')
print(f'Pool lr={POOL_LR}  bs={POOL_BS}  backbone={BACKBONE}  dataset={DATASET}')

# ── Stale-checkpoint safety warning ──────────────────────────────────────
existing = sorted(f for f in os.listdir(POOL_DIR) if f.endswith('.pt'))
if existing:
    print(f"WARNING: {len(existing)} existing pool checkpoints found in {POOL_DIR}")
    print("If you changed POOL_LR or any pool hyperparameter since these were saved,")
    print("delete them manually from Drive before continuing, then re-run this cell.")
    print(f"Proceeding with resumable logic — starting from checkpoint index {len(existing)}")
start_idx = len(existing)

if start_idx >= N_POOL:
    print('Pool already complete.')
else:
    # D_f with augmentation (deviates from NegMerge Appendix A — needed for
    # pool diversity; without aug all 27 members collapse to the same vector).
    forget_loader_aug = make_loader(trainset_aug, indices=forget_indices,
                                    batch_size=POOL_BS, shuffle=True, num_workers=2)
    # Eval still uses the plain (non-aug) transform for a fair D_f accuracy read.
    df_eval_loader    = make_loader(trainset_plain, indices=forget_indices,
                                    batch_size=512, shuffle=False)

    for i in range(start_idx, N_POOL):
        epochs, wd, ls = grid[i]
        set_seed(100 + i)
        print(f'[{i+1}/{N_POOL}] backbone={BACKBONE} dataset={DATASET} '
              f'lr={POOL_LR} epochs={epochs} wd={wd} ls={ls}')
        t0 = time.time()
        model = finetune_pool_member(
            theta_pre_state, forget_loader_aug,
            epochs=epochs, lr=POOL_LR, momentum=0.9,
            weight_decay=wd, label_smoothing=ls,
        )
        df_acc = evaluate_accuracy(model, df_eval_loader)
        print(f'  trained in {(time.time()-t0)/60:.1f} min  Acc D_f = {df_acc:.2f}%')

        cp_path = os.path.join(POOL_DIR, f'model_{i:02d}.pt')
        torch.save({
            'state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
            'backbone': BACKBONE, 'dataset': DATASET,
            'epochs': epochs, 'weight_decay': wd, 'label_smoothing': ls,
            'pool_lr': POOL_LR, 'df_acc_train_end': df_acc,
        }, cp_path)
        print(f'  saved → {cp_path}')

        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print('Pool training complete.')


Pool grid size: 27  (REDUCED_POOL=False)
Pool lr=0.1  bs=256  backbone=resnet18  dataset=cifar10
If you changed POOL_LR or any pool hyperparameter since these were saved,
delete them manually from Drive before continuing, then re-run this cell.
Proceeding with resumable logic — starting from checkpoint index 27
Pool already complete.
Pool training complete.


## Cell 9 — [Stage 4] Streaming τ_merged + τ_uniform

Computes both in a single pass over pool checkpoints: load one checkpoint, accumulate sign-consensus mask and running sum, discard, repeat. Saves both τ_merged (NegMerge) and τ_uniform (Baseline).

Architecture-agnostic: works for any backbone because it operates on raw state_dict tensors.

In [18]:
# Magnitude threshold: enabled now that the pool is diverse (guide §5.4).
USE_MAGNITUDE_THRESHOLD       = True    # zero out tiny-|τ| entries in the agree_mask
MAGNITUDE_THRESHOLD_PERCENTILE = 10     # zero out the bottom N% by mean |τ|

def load_pool_sd(path):
    """Load a pool checkpoint and return its state_dict, whether it was saved
    wrapped ({'state_dict': ..., 'epochs': ..., ...}) or as a bare state_dict."""
    raw = torch.load(path, map_location='cpu')
    return raw['state_dict'] if isinstance(raw, dict) and 'state_dict' in raw else raw


pool_files = sorted(os.path.join(POOL_DIR, f) for f in os.listdir(POOL_DIR)
                    if f.startswith('model_') and f.endswith('.pt'))
n_pool = len(pool_files)
print(f'Merging over {n_pool} pool checkpoints  (backbone={BACKBONE})')
assert n_pool > 0, 'No pool checkpoints found. Run Cell 8 first.'

# Exclude BN running stats — these are accumulated dataset statistics,
# not learnable parameters. Modifying them with a D_f-biased task vector
# corrupts running_var (can go negative → sqrt(negative) = NaN in eval mode).
# NegMerge's CLIP scenario uses ViT/LayerNorm so this never appeared there.
param_keys = [
    k for k, v in theta_pre_state.items()
    if v.dtype.is_floating_point
    and 'running_mean' not in k
    and 'running_var'  not in k
    and 'num_batches_tracked' not in k
]
print(f'Merging {len(param_keys)} float parameter tensors.')
print(f'USE_MAGNITUDE_THRESHOLD={USE_MAGNITUDE_THRESHOLD}  '
      f'(percentile={MAGNITUDE_THRESHOLD_PERCENTILE})')

tau_sum     = {k: torch.zeros_like(theta_pre_state[k], dtype=torch.float32) for k in param_keys}
tau_abs_sum = {k: torch.zeros_like(theta_pre_state[k], dtype=torch.float32) for k in param_keys}
first_sign  = None
agree_mask  = None

for i, path in enumerate(pool_files):
    ft_sd = load_pool_sd(path)
    tv    = {k: (ft_sd[k].to(torch.float32) - theta_pre_state[k].to(torch.float32))
             for k in param_keys}
    signs = {k: torch.sign(v) for k, v in tv.items()}
    if i == 0:
        first_sign = signs
        agree_mask = {k: torch.ones_like(v, dtype=torch.bool) for k, v in tv.items()}
    else:
        for k in agree_mask:
            agree_mask[k] = agree_mask[k] & (signs[k] == first_sign[k])
    for k in tau_sum:
        tau_sum[k]     += tv[k]
        tau_abs_sum[k] += tv[k].abs()
    del ft_sd, tv, signs
    if (i + 1) % 5 == 0 or i == n_pool - 1:
        print(f'  merged {i+1}/{n_pool}')

# tau_uniform = mean of all task vectors (no masking)
tau_uniform = {k: tau_sum[k] / n_pool for k in param_keys}

# Optionally further gate by magnitude: zero out bottom P% by mean |τ|
if USE_MAGNITUDE_THRESHOLD:
    print(f'Applying magnitude threshold at {MAGNITUDE_THRESHOLD_PERCENTILE}th percentile...')
    # Use mean |τ_k| across pool members, not |mean(τ_k)|. The latter cancels
    # when members partially disagree in sign, understating true magnitude.
    for k in param_keys:
        mean_abs = tau_abs_sum[k] / n_pool
        thr = torch.quantile(mean_abs.flatten(), MAGNITUDE_THRESHOLD_PERCENTILE / 100.0)
        agree_mask[k] = agree_mask[k] & (mean_abs > thr)

tau_merged = {k: tau_uniform[k] * agree_mask[k].to(torch.float32) for k in param_keys}


torch.save(tau_merged,  TAU_MERGED_PATH)
torch.save(tau_uniform, TAU_UNIFORM_PATH)
print(f'Saved tau_merged  -> {TAU_MERGED_PATH}')
print(f'Saved tau_uniform -> {TAU_UNIFORM_PATH}')

# Sparsity report
total    = sum(t.numel() for t in tau_merged.values())
zeros    = sum((t == 0).sum().item() for t in tau_merged.values())
sparsity = 100.0 * zeros / total
print(f'tau_merged global sparsity: {sparsity:.2f}%  (expected 85-93%)')
if sparsity > 95.0:
    print('WARNING: sparsity > 95% — consider lowering MAGNITUDE_THRESHOLD_PERCENTILE '
          '(lower = fewer zeros) or setting USE_MAGNITUDE_THRESHOLD=False.')

from collections import defaultdict
def _layer_group(key):
    if key in ('conv1.weight', 'bn1.weight', 'bn1.bias') or key.startswith('features.0'):
        return 'stem'
    if key.startswith('linear') or key.startswith('fc') or 'classifier' in key:
        return 'classifier'
    return 'body'

by_grp = defaultdict(lambda: [0, 0])
for k, t in tau_merged.items():
    grp = _layer_group(k)
    by_grp[grp][0] += (t == 0).sum().item()
    by_grp[grp][1] += t.numel()
for grp, (z, n) in sorted(by_grp.items()):
    print(f'  {grp:12s}: {100*z/n:.2f}% zeros  ({n} params)')


# ── NaN sanity check at α=0.05 (the smallest non-zero α in ALPHAS) ─────────
_verify_state = {k: v.clone() for k, v in theta_pre_state.items()}
for k in tau_merged:
    _verify_state[k] = (theta_pre_state[k].to(torch.float32) - 0.05 * tau_merged[k]
                        ).to(theta_pre_state[k].dtype)
_m = make_model().to(DEVICE); _m.load_state_dict(_verify_state)
_m.eval()
_probe_loader = make_loader(testset, batch_size=4, shuffle=False)
_probe_x = next(iter(_probe_loader))[0].to(DEVICE)
with torch.no_grad():
    _probe_out = _m(_probe_x)
assert not torch.isnan(_probe_out).any() and not torch.isinf(_probe_out).any(), (
    'Unexpected NaN/Inf at α=0.05 — BN running stats should now be excluded from τ_merged. Check that param_keys dropped running_mean / running_var / num_batches_tracked.'
)
print('No NaN at α=0.05')
del _m, _verify_state, _probe_x, _probe_out
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


Merging over 27 pool checkpoints  (backbone=resnet18)
Merging 62 float parameter tensors.
USE_MAGNITUDE_THRESHOLD=True  (percentile=10)
  merged 5/27
  merged 10/27
  merged 15/27
  merged 20/27
  merged 25/27
  merged 27/27
Applying magnitude threshold at 10th percentile...
Saved tau_merged  -> /content/drive/MyDrive/negmerge_reproduction/resnet18_cifar10/tau_merged.pt
Saved tau_uniform -> /content/drive/MyDrive/negmerge_reproduction/resnet18_cifar10/tau_uniform.pt
tau_merged global sparsity: 99.32%  (expected 85-93%)
  body        : 99.33% zeros  (11166976 params)
  classifier  : 99.92% zeros  (5130 params)
  stem        : 68.97% zeros  (1856 params)
No NaN at α=0.05


## Cell 10 — MIA-Efficacy (SVC on softmax max-prob)

Attacker trained on D_r (member) vs D_test (non-member) confidences, evaluated on D_f. The attacker-training split is fixed once here and reused by every method (Cells 11–13). Architecture-agnostic.

In [19]:
def get_confidences(model, loader, device=DEVICE):
    """Max softmax probability for each sample. NaN/Inf-guarded."""
    model.eval()
    confs = []
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device, non_blocking=True)
            logits = model(x)
            logits = torch.nan_to_num(logits, nan=0.0, posinf=1e6, neginf=-1e6)
            probs  = torch.softmax(logits, dim=1)
            confs.append(probs.max(dim=1).values.cpu())
    confs = torch.cat(confs).numpy()
    return np.nan_to_num(confs, nan=0.5)


def mia_efficacy(model, D_r_loader=None, D_test_loader=None, D_f_loader=None, device=DEVICE):
    """
    MIA-Efficacy from Jia et al. 2023 Appendix C.3.
    Threshold attacker on D_r (member) vs D_test (non-member) confidences.
    Returns % of D_f samples classified as non-member.  Higher = better unlearning.
    Target for Retrain baseline: ~10-20%.
    """
    # Fall back to module-level fixed loaders when callers pass None
    r_ldr  = D_r_loader   if D_r_loader   is not None else mia_retain_loader
    t_ldr  = D_test_loader if D_test_loader is not None else mia_nonmember_loader
    f_ldr  = D_f_loader   if D_f_loader   is not None else mia_forget_loader

    c_m = get_confidences(model, r_ldr,  device)
    c_n = get_confidences(model, t_ldr,  device)
    c_f = get_confidences(model, f_ldr,  device)

    # Balance attacker training
    n     = min(len(c_m), len(c_n))
    rng   = np.random.default_rng(42)
    c_m   = c_m[rng.choice(len(c_m), n, replace=False)]
    c_n   = c_n[rng.choice(len(c_n), n, replace=False)]

    # Find optimal threshold
    best_acc, best_thr = 0.0, 0.5
    for thr in np.linspace(0.0, 1.0, 101):
        tp  = (c_m > thr).sum()
        tn  = (c_n <= thr).sum()
        acc = (tp + tn) / (2 * n)
        if acc > best_acc:
            best_acc, best_thr = acc, float(thr)

    tn_forget = (c_f <= best_thr).sum()
    efficacy  = 100.0 * tn_forget / len(c_f)
    print(f"  MIA threshold={best_thr:.2f}  attacker_acc={best_acc*100:.1f}%  "
          f"MIA-Efficacy={efficacy:.2f}%")
    return efficacy


# Fix the attacker-training loaders ONCE so every method uses the same samples.
MIA_N = 10_000
_rng = np.random.default_rng(123)
mia_retain_idx    = _rng.choice(retain_indices, size=MIA_N, replace=False)
mia_nonmember_idx = np.arange(len(testset))

mia_retain_loader    = make_loader(trainset_plain, indices=mia_retain_idx,    batch_size=512, shuffle=False)
mia_nonmember_loader = make_loader(testset,        indices=mia_nonmember_idx, batch_size=512, shuffle=False)
mia_forget_loader    = make_loader(trainset_plain, indices=forget_indices,    batch_size=512, shuffle=False)

# Validate on reference checkpoints
_m = make_model().to(DEVICE)
_m.load_state_dict(theta_pre_state)
mia_pre = mia_efficacy(_m)
print(f'MIA on theta_pre:     {mia_pre:.2f}%  (expect < 5 — theta_pre has seen D_f)')
_m.load_state_dict(theta_retrain_state)
mia_ret = mia_efficacy(_m)
print(f'MIA on theta_retrain: {mia_ret:.2f}%  (expect 10-20)')
del _m; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


  MIA threshold=0.99  attacker_acc=55.9%  MIA-Efficacy=1.24%
MIA on theta_pre:     1.24%  (expect < 5 — theta_pre has seen D_f)
  MIA threshold=0.99  attacker_acc=56.4%  MIA-Efficacy=14.84%
MIA on theta_retrain: 14.84%  (expect 10-20)


## Cell 11 — [Stage 5] NegMerge α search + evaluation

α ∈ {0.0, 0.05, …, 1.0}. Builds θ_unlearn = θ_pre − α·τ_merged for each α, evaluates Acc D_r / D_f / D_test / MIA, picks α minimising Avg. Gap vs Retrain.

In [20]:
ALPHAS = np.round(np.arange(0.0, 1.01, 0.05), 4)

eval_loaders = {
    'dr':    make_loader(trainset_plain, indices=retain_indices, batch_size=512, shuffle=False),
    'df':    make_loader(trainset_plain, indices=forget_indices, batch_size=512, shuffle=False),
    'dtest': make_loader(testset, batch_size=512, shuffle=False),
}

# Build Retrain reference (use our own run, not paper values — guide §6.3)
_m = make_model().to(DEVICE); _m.load_state_dict(theta_retrain_state)
retrain_metrics = {
    'acc_dr':    evaluate_accuracy(_m, eval_loaders['dr']),
    'acc_df':    evaluate_accuracy(_m, eval_loaders['df']),
    'acc_dtest': evaluate_accuracy(_m, eval_loaders['dtest']),
    'mia':       mia_efficacy(_m),
}
print('Retrain reference:', {k: f'{v:.2f}' for k, v in retrain_metrics.items()})
del _m; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


def apply_tau(tau, alpha):
    out = {k: v.clone() for k, v in theta_pre_state.items()}
    for k in tau:
        out[k] = (theta_pre_state[k].to(torch.float32) - alpha * tau[k]).to(theta_pre_state[k].dtype)
    return out


def is_model_degenerate(model, loader, device=DEVICE):
    """Returns True if the first batch produces NaN or Inf outputs."""
    model.eval()
    x = next(iter(loader))[0][:8].to(device)
    with torch.no_grad():
        out = model(x)
    return bool(torch.isnan(out).any() or torch.isinf(out).any())


def eval_state(state):
    m = make_model().to(DEVICE); m.load_state_dict(state)
    out = {
        'acc_dr':    evaluate_accuracy(m, eval_loaders['dr']),
        'acc_df':    evaluate_accuracy(m, eval_loaders['df']),
        'acc_dtest': evaluate_accuracy(m, eval_loaders['dtest']),
        'mia':       mia_efficacy(m),
    }
    del m; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return out


def avg_gap(metrics, ref):
    return float(np.mean([abs(metrics[k] - ref[k]) for k in ('acc_dr','acc_df','acc_dtest','mia')]))


def sweep(tau, tau_name):
    rows = []
    _probe_loader = eval_loaders['dtest']
    for a in ALPHAS:
        state = apply_tau(tau, a)
        # NaN guard: skip degenerate alpha values rather than crashing
        _chk = make_model().to(DEVICE); _chk.load_state_dict(state)
        if is_model_degenerate(_chk, _probe_loader):
            print(f'  alpha={a:.2f}  SKIPPED (degenerate model — NaN/Inf outputs)')
            del _chk; continue
        del _chk; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        m   = eval_state(state)
        gap = avg_gap(m, retrain_metrics)
        rows.append({'method': tau_name, 'alpha': float(a), **m, 'avg_gap': gap})
        print(f'  alpha={a:.2f}  D_r={m["acc_dr"]:.2f} D_f={m["acc_df"]:.2f} '
              f'D_test={m["acc_dtest"]:.2f} MIA={m["mia"]:.2f}  gap={gap:.3f}')
    return rows


print('--- NegMerge sweep ---')
negmerge_rows = sweep(tau_merged, 'NegMerge')
if negmerge_rows:
    best_neg = min(negmerge_rows, key=lambda r: r['avg_gap'])
    print(f'Best NegMerge: alpha={best_neg["alpha"]:.2f}  Avg Gap={best_neg["avg_gap"]:.3f}  '
          f'(paper target approx {targets["negmerge_avg_gap"]})')
else:
    print('WARNING: all alpha values were skipped (degenerate model). Check tau_merged.')


  MIA threshold=0.99  attacker_acc=56.4%  MIA-Efficacy=14.84%
Retrain reference: {'acc_dr': '100.00', 'acc_df': '94.40', 'acc_dtest': '94.43', 'mia': '14.84'}
--- NegMerge sweep ---
  MIA threshold=0.99  attacker_acc=55.9%  MIA-Efficacy=1.24%
  alpha=0.00  D_r=100.00 D_f=100.00 D_test=95.01 MIA=1.24  gap=4.945
  MIA threshold=0.99  attacker_acc=55.9%  MIA-Efficacy=1.10%
  alpha=0.05  D_r=100.00 D_f=100.00 D_test=95.00 MIA=1.10  gap=4.977
  MIA threshold=0.99  attacker_acc=55.8%  MIA-Efficacy=1.26%
  alpha=0.10  D_r=100.00 D_f=100.00 D_test=94.91 MIA=1.26  gap=4.916
  MIA threshold=0.99  attacker_acc=55.7%  MIA-Efficacy=1.96%
  alpha=0.15  D_r=99.93 D_f=99.94 D_test=94.57 MIA=1.96  gap=4.657
  MIA threshold=0.99  attacker_acc=55.4%  MIA-Efficacy=3.46%
  alpha=0.20  D_r=99.71 D_f=99.78 D_test=94.13 MIA=3.46  gap=4.338
  MIA threshold=0.99  attacker_acc=54.9%  MIA-Efficacy=6.60%
  alpha=0.25  D_r=99.04 D_f=99.00 D_test=93.34 MIA=6.60  gap=3.721
  MIA threshold=0.99  attacker_acc=54.2%  MI

## Cell 12 — [Baseline] Uniform Merge

In [21]:
print('--- Uniform Merge sweep ---')
uniform_rows   = sweep(tau_uniform, 'UniformMerge')
best_uniform   = min(uniform_rows, key=lambda r: r['avg_gap'])
paper_uni_gap  = targets.get('uniform_merge_avg_gap')
paper_note     = f'paper target ≈ {paper_uni_gap}' if paper_uni_gap else 'not reported in paper'
print(f'Best Uniform: α={best_uniform["alpha"]:.2f}  Avg Gap={best_uniform["avg_gap"]:.3f}  ({paper_note})')


--- Uniform Merge sweep ---
  MIA threshold=0.99  attacker_acc=55.9%  MIA-Efficacy=1.24%
  alpha=0.00  D_r=100.00 D_f=100.00 D_test=95.01 MIA=1.24  gap=4.945
  MIA threshold=0.99  attacker_acc=56.0%  MIA-Efficacy=3.54%
  alpha=0.05  D_r=99.99 D_f=99.94 D_test=94.68 MIA=3.54  gap=4.274
  MIA threshold=0.99  attacker_acc=55.5%  MIA-Efficacy=10.30%
  alpha=0.10  D_r=99.57 D_f=98.80 D_test=93.52 MIA=10.30  gap=2.569
  MIA threshold=0.99  attacker_acc=53.9%  MIA-Efficacy=25.84%
  alpha=0.15  D_r=96.32 D_f=94.46 D_test=89.85 MIA=25.84  gap=4.831
  MIA threshold=0.99  attacker_acc=52.2%  MIA-Efficacy=51.02%
  alpha=0.20  D_r=85.36 D_f=82.12 D_test=80.10 MIA=51.02  gap=19.357
  MIA threshold=0.86  attacker_acc=51.2%  MIA-Efficacy=48.88%
  alpha=0.25  D_r=64.91 D_f=61.20 D_test=61.61 MIA=48.88  gap=33.787
  MIA threshold=0.81  attacker_acc=51.1%  MIA-Efficacy=66.94%
  alpha=0.30  D_r=41.67 D_f=38.68 D_test=40.05 MIA=66.94  gap=55.133
  MIA threshold=0.62  attacker_acc=50.6%  MIA-Efficacy=53.46%